In [169]:
#! python -m pip install numpy scipy matplotlib
import scipy as scp
import numpy as np
import matplotlib.pyplot as plt

In [170]:
csi_largo = np.random.randn(500, 64) + 1j * np.random.randn(500, 64)

In [171]:
csi_ex = np.array([
    # Paquete 1 (t = 0)
    [13.2 + 7.2j,  5.4 + 14.0j, -6.2 + 13.6j, -14.1 + 5.0j,
    -13.4 - 6.6j, -4.6 - 14.3j,  7.0 - 13.3j,  14.4 - 4.2j],
    
    # Paquete 2 (t = 1)
    [11.7 + 9.3j,  3.2 + 14.6j, -8.3 + 12.5j, -14.7 + 3.0j,
    -11.8 - 9.2j, -2.5 - 14.8j,  8.8 - 12.1j,  14.8 - 2.3j],
    
    # Paquete 3 (t = 2)
    [ 9.9 + 11.2j,  0.9 + 15.0j, -10.2 + 11.0j, -14.9 + 0.9j,
    -9.9 - 11.2j, -0.3 - 15.0j,  10.4 - 10.8j,  14.9 - 0.3j]
])

fases = []


In [172]:
for subportadora in csi_largo:
    fases_z =[]
    for z in subportadora:
        I = z.real  # parte real
        Q = z.imag # parte imaginaria 
        fase = np.arctan2(Q, I) 
        fases_z.append(fase)
        
    fases.append(fases_z)
fases_brutas = np.array(fases)
fases_unwrapped = np.unwrap(fases_brutas, axis=0) 
print(fases_unwrapped)

[[  1.89289582   2.97400904   1.11059025 ...  -0.52009184   0.18151064
    1.46827372]
 [  3.75308953   1.75062512  -0.46492776 ...  -1.62754737  -0.50789062
    2.55863792]
 [  5.30430765   3.74141317   0.28399805 ...  -2.82959979  -1.31670795
    1.00301691]
 ...
 [ 85.68225022  27.23487342 -39.54244648 ...  47.26393772   4.01655204
   -5.25994493]
 [ 87.90644779  27.51688194 -42.22450106 ...  46.40404554   4.46822314
   -7.42972906]
 [ 87.95939074  28.37161333 -39.14923514 ...  48.36757128   6.99499987
  -10.42703392]]


In [173]:
fs = 100 #asumo que la frecuencia de lo que me mande hardware será 100 paquetes x seg
nyquist = fs / 2 # tiene que ser la mitad de lo que recibe pq si no tosquea x alguna razon. se llama limite de nyquist
frecuencia_baja, frecuencia_alta = 0.1, 0.5 # 0.1 son 6rpm y 0.5 30rpm
low = frecuencia_baja / nyquist
high = frecuencia_alta / nyquist
b, a = scp.signal.butter(N=2, Wn=[low, high], btype='bandpass') #plantilla del filtro
fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0) #aplico el filtro a mi fase
print(fases_filtradas)


[[-2.90588108  4.19932275  0.06988379 ... -0.72300363  4.22339439
  -6.37355344]
 [-3.03094247  3.99852847 -0.04173114 ... -0.75579358  4.21660911
  -6.59524641]
 [-3.15683093  3.79600723 -0.15268652 ... -0.78878692  4.2083891
  -6.81770756]
 ...
 [ 0.09261966  0.04066418 -0.03307978 ...  0.0472433   0.09116766
  -0.05571623]
 [ 0.07764684  0.03426886 -0.0287538  ...  0.03983072  0.07728478
  -0.04656838]
 [ 0.06433144  0.02853792 -0.02467658 ...  0.03319105  0.06474459
  -0.03846289]]


In [ ]:
varianzas = np.var(fases_filtradas, axis=0) #calcula varianza
mejor_subportadora = np.argmax(varianzas) #agarro la subportadora de + varianza
mejor_señal = fases_filtradas[:, mejor_subportadora]

# 2. FFT con Zero-Padding (n_fft = 10000 para dar resolución fina en Hz)
n_fft = 10000 
fft_valores = np.fft.fft(mejor_señal, n=n_fft)
magnitudes = np.abs(fft_valores)
frecuencias = np.fft.fftfreq(n_fft, d=1/fs)

# 3. Mapear a frecuencias positivas y a RPM
mitad = n_fft // 2
frecuencias_pos = frecuencias[:mitad]
magnitudes_pos = magnitudes[:mitad]
rpm_pos = frecuencias_pos * 60.0

# 4. Máscara booleana para el rango de respiración humana (6 a 30 RPM)
mascara_humana = (rpm_pos >= 6.0) & (rpm_pos <= 30.0)
rpm_validas = rpm_pos[mascara_humana]
magnitudes_validas = magnitudes_pos[mascara_humana]

# 5. Detección de presencia y cálculo de RPM
if len(magnitudes_validas) > 0:
    indice_pico = np.argmax(magnitudes_validas)
    pico_potencia = magnitudes_validas[indice_pico]
    promedio_ruido = np.mean(magnitudes_pos)
    
    # Criterio: el pico debe destacar sobre el ruido de fondo
    if pico_potencia > (3.0 * promedio_ruido):
        rpm_detectadas = rpm_validas[indice_pico]
        print(f"Presencia detectada: Ritmo: {rpm_detectadas:.1f} RPM (Subportadora {mejor_subportadora})")
    else:
        print("Sin presencia humana detectada ")


Presencia detectada: Ritmo: 14.4 RPM (Subportadora 40)
